# 04: KNN for Graph-Powered RAG

This notebook demonstrates advanced graph-powered RAG using:
1. Graph Data Science (GDS) for node embeddings
2. K-Nearest Neighbors (KNN) to find similar entities
3. Graph ML-enhanced retrieval for better recommendations

## Overview

We'll:
1. Create node embeddings using FastRP
2. Use KNN to find similar entities
3. Build graph-enhanced retrieval for RAG
4. Combine with vector search for comprehensive results


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
from graphdatascience import GraphDataScience
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores.neo4j_vector import Neo4jVector
from langchain.graphs import Neo4jGraph

print("✅ Additional libraries imported")


In [ ]:
# Settings, dependencies, driver, and neo4j_manager are already initialized in 00-import.ipynb
# Create GDS client (set AURA_DS=True if using AuraDS)
AURA_DS = False
gds = GraphDataScience(
    settings.neo4j_uri,
    auth=(settings.neo4j_username, settings.neo4j_password),
    aura_ds=AURA_DS
)
gds.set_database(settings.neo4j_database)

# Create Neo4jGraph for queries
kg = Neo4jGraph(
    url=settings.neo4j_uri,
    username=settings.neo4j_username,
    password=settings.neo4j_password,
    database=settings.neo4j_database
)

print(f"✅ GDS and Neo4jGraph initialized")
try:
    print(f"✅ GDS version: {gds.version()}")
except Exception as e:
    print(f"⚠️  GDS not available: {e}")


## Part 1: Create Node Embeddings

Use FastRP to create node embeddings based on graph structure.


In [ ]:
# Clear any existing graph projections
def clear_all_graphs():
    """Clear all existing graph projections."""
    try:
        g_names = gds.graph.list().graphName.tolist()
        for g_name in g_names:
            g = gds.graph.get(g_name)
            g.drop()
        print(f"Cleared {len(g_names)} existing graph projections")
    except Exception as e:
        print(f"No existing graphs to clear: {e}")

clear_all_graphs()


In [ ]:
# Project graph: Create co-occurrence graph based on entities appearing in same notes
# This creates relationships between entities that appear together
print("Creating graph projection...")

gds.run_cypher('''
  MATCH (e1:Entity)<-[:CONTAINS]-(:Note)-[:CONTAINS]->(e2:Entity)
  WHERE e1 <> e2
  WITH gds.graph.project("entity_cograph", e1, e2,
       {relationshipType: "CO_OCCURS_WITH"}) AS g
  RETURN g.graphName
''')

g = gds.graph.get("entity_cograph")
print(f"✅ Created graph projection: {g.name()}")
print(f"   Nodes: {g.node_count()}")
print(f"   Relationships: {g.relationship_count()}")


In [ ]:
# Create FastRP node embeddings
print("Creating node embeddings with FastRP...")

gds.fastRP.mutate(
    g, 
    mutateProperty='nodeEmbedding', 
    embeddingDimension=128, 
    randomSeed=7474, 
    concurrency=4, 
    iterationWeights=[0.0, 1.0, 1.0]
)

# Write embeddings back to database
gds.graph.writeNodeProperties(g, ['nodeEmbedding'])

print("✅ Node embeddings created and written to database")


## Part 2: K-Nearest Neighbors

Use KNN to find similar entities based on node embeddings.


In [ ]:
# Clear any existing KNN relationships
gds.run_cypher('''
    MATCH(:Entity)-[r:SIMILAR_TO]->()
    CALL {
        WITH r
        DELETE r
    } IN TRANSACTIONS OF 1000 ROWS
''')

print("Cleared existing SIMILAR_TO relationships")


In [ ]:
# Run KNN to find similar entities
print("Running KNN algorithm...")

knn_stats = gds.knn.write(
    g, 
    nodeProperties=['nodeEmbedding'],
    writeRelationshipType='SIMILAR_TO', 
    writeProperty='similarity',
    sampleRate=1.0, 
    initialSampler='randomWalk', 
    concurrency=1, 
    similarityCutoff=0.5,  # Only keep relationships with similarity >= 0.5
    randomSeed=7474
)

# Clear graph projection
g.drop()

print("✅ KNN relationships created")
print(f"   Relationships written: {knn_stats.get('relationshipsWritten', 'N/A')}")
print(f"   Nodes compared: {knn_stats.get('nodesCompared', 'N/A')}")


## Part 3: Graph-Enhanced Retrieval

Use KNN relationships to enhance retrieval with graph context.


In [ ]:
# Create custom embeddings for LiteLLM proxy
class ProxyEmbeddings(OpenAIEmbeddings):
    """Custom embeddings class that uses LiteLLM proxy."""
    
    def __init__(self, proxy_base_url: str, api_key: str, model: str, **kwargs):
        api_url = f"{proxy_base_url}/v1"
        super().__init__(
            openai_api_base=api_url,
            openai_api_key=api_key,
            model=model,
            **kwargs
        )

proxy_base_url = f"http://{settings.litellm_proxy_host}:{settings.litellm_proxy_port}"
embedding_model = ProxyEmbeddings(
    proxy_base_url=proxy_base_url,
    api_key=settings.openai_api_key or "",
    model=settings.litellm_proxy_embedding_model,
)

# Create enhanced vector search with KNN relationships
kg_enhanced_search = Neo4jVector.from_existing_index(
    embedding=embedding_model,
    url=settings.neo4j_uri,
    username=settings.neo4j_username,
    password=settings.neo4j_password,
    database=settings.neo4j_database,
    index_name=settings.neo4j_vector_index_name,
    retrieval_query="""
    WITH node AS note, score AS searchScore
    
    // Find entities in this note
    OPTIONAL MATCH (note)-[:CONTAINS]->(e:Entity)
    
    // Find similar entities using KNN relationships
    OPTIONAL MATCH (e)-[:SIMILAR_TO]-(similar:Entity)
    
    // Find notes containing similar entities
    OPTIONAL MATCH (similar)<-[:CONTAINS]-(related:Note)
    WHERE related <> note
    
    WITH note, searchScore, 
         count(DISTINCT related) AS relatedNoteCount,
         avg(similarity) AS avgSimilarity
    
    RETURN note.text AS text,
           searchScore,
           relatedNoteCount,
           avgSimilarity,
           {file_path: note.file_path, 
            file_name: note.file_name,
            related_notes: relatedNoteCount,
            avg_similarity: avgSimilarity} AS metadata
    ORDER BY relatedNoteCount DESC, avgSimilarity DESC, searchScore DESC
    LIMIT 10
    """
)

print("✅ Enhanced search with KNN configured")


In [ ]:
# Perform enhanced search
search_prompt = "project ideas and development tasks"

enhanced_results = kg_enhanced_search.similarity_search(search_prompt, k=5)

print(f"Enhanced KNN results for: '{search_prompt}'\n")
for i, doc in enumerate(enhanced_results, 1):
    print(f"{i}. {doc.page_content[:200]}...")
    if hasattr(doc, 'metadata') and doc.metadata:
        print(f"   Related notes: {doc.metadata.get('related_notes', 0)}")
        print(f"   Avg similarity: {doc.metadata.get('avg_similarity', 0):.3f}")
        print(f"   File: {doc.metadata.get('file_path', 'N/A')}\n")


In [ ]:
# Find similar entities for a given entity
entity_name = "Project"  # Change this to an entity from your graph

query = """
MATCH (e:Entity {name: $entity_name})-[r:SIMILAR_TO]-(similar:Entity)
RETURN similar.name AS similar_entity,
       similar.type AS entity_type,
       r.similarity AS similarity_score
ORDER BY r.similarity DESC
LIMIT 10
"""

results = kg.query(query, params={"entity_name": entity_name})

if results:
    print(f"Entities similar to '{entity_name}':\n")
    df = pd.DataFrame(results)
    print(df)
else:
    print(f"No similar entities found for '{entity_name}'")
    print("Try a different entity name from your graph")


## Summary

This notebook demonstrated:
1. **Node Embeddings**: Using FastRP to encode graph structure
2. **KNN Relationships**: Finding similar entities based on embeddings
3. **Enhanced Retrieval**: Combining vector search with KNN relationships
4. **Graph-Powered RAG**: Using graph ML to improve search results

The combination of vector search, graph patterns, and graph ML provides a powerful foundation for knowledge retrieval and question answering.
